**Note:** This is the original Colab notebook. The maintained pipeline is `code/Solar.py`, `code/inverter_fault_detection.py` and `code/make_dashboard.py`. Outputs here were cleared after a units fix (AC_POWER is in kW, so energy is kW x 0.25 h). Re-run the cells to regenerate them.

In [ ]:
"""
Solar Asset Performance Monitoring & Analysis
==============================================
Author: Vinayak Sonter
Purpose: Ingest, clean, merge, and analyze solar generation and weather data
         to calculate performance metrics, identify underperformance, and
         quantify lost energy and revenue impact.

Data Sources:
    - Plant 1 & 2 Generation Data (Kaggle - Solar Power Generation)
    - Plant 1 & 2 Weather Sensor Data (Kaggle)
    - NREL Rooftop PV Technical Potential (US)
"""

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load raw files
gen1 = pd.read_csv('/content/Plant_1_Generation_Data.csv')
gen2 = pd.read_csv('/content/Plant_2_Generation_Data.csv')
wx1 = pd.read_csv('/content/Plant_1_Weather_Sensor_Data.csv')
wx2 = pd.read_csv('/content/Plant_2_Weather_Sensor_Data.csv')

# Fix date formats
gen1['DATE_TIME'] = pd.to_datetime(gen1['DATE_TIME'], format='%d-%m-%Y %H:%M')
gen2['DATE_TIME'] = pd.to_datetime(gen2['DATE_TIME'])
wx1['DATE_TIME'] = pd.to_datetime(wx1['DATE_TIME'])
wx2['DATE_TIME'] = pd.to_datetime(wx2['DATE_TIME'])

# Add helper columns
for df in [gen1, gen2, wx1, wx2]:
    df['DATE'] = df['DATE_TIME'].dt.date
    df['HOUR'] = df['DATE_TIME'].dt.hour

gen1['PLANT_LABEL'] = 'Plant_1'
gen2['PLANT_LABEL'] = 'Plant_2'
wx1['PLANT_LABEL'] = 'Plant_1'
wx2['PLANT_LABEL'] = 'Plant_2'

gen_all = pd.concat([gen1, gen2], ignore_index=True)
wx_all = pd.concat([wx1, wx2], ignore_index=True)

print(f"Generation: {len(gen_all):,} rows")
print(f"Weather: {len(wx_all):,} rows")
print(f"Date range: {gen_all['DATE_TIME'].min()} → {gen_all['DATE_TIME'].max()}")
print(f"Plants: {gen_all['PLANT_LABEL'].unique().tolist()}")
print(f"Inverters: {gen_all['SOURCE_KEY'].nunique()}")

In [ ]:
# Merge generation with weather on timestamp and plant
merged = pd.merge(
    gen_all,
    wx_all[['DATE_TIME', 'PLANT_ID', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']],
    on=['DATE_TIME', 'PLANT_ID'],
    how='left'
)

# Forward fill minor weather gaps
merged[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']] = \
    merged.groupby('PLANT_ID')[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']].ffill()

# Drop remaining nulls
before = len(merged)
merged = merged.dropna(subset=['IRRADIATION'])
after = len(merged)

print(f"Merged rows: {len(merged):,}")
print(f"Dropped: {before - after} rows with missing irradiation")
print(f"Null check:")
for col in ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']:
    nulls = merged[col].isnull().sum()
    print(f"  {col}: {nulls}")

In [ ]:
# Filter to daylight hours only (irradiation > 0)
daylight = merged[merged['IRRADIATION'] > 0].copy()
print(f"Daylight records: {len(daylight):,} out of {len(merged):,}")

# DC to AC conversion efficiency
daylight['DC_AC_RATIO'] = np.where(
    daylight['DC_POWER'] > 0,
    daylight['AC_POWER'] / daylight['DC_POWER'],
    0
)

# Daily aggregation per inverter
daily_inverter = daylight.groupby(['DATE', 'PLANT_LABEL', 'PLANT_ID', 'SOURCE_KEY']).agg(
    total_dc_kwh=('DC_POWER', 'sum'),
    total_ac_kwh=('AC_POWER', 'sum'),
    daily_yield=('DAILY_YIELD', 'max'),
    avg_irradiance=('IRRADIATION', 'mean'),
    total_irradiance=('IRRADIATION', 'sum'),
    avg_ambient_temp=('AMBIENT_TEMPERATURE', 'mean'),
    avg_module_temp=('MODULE_TEMPERATURE', 'mean'),
    avg_dc_ac_ratio=('DC_AC_RATIO', 'mean'),
    records_count=('AC_POWER', 'count')
).reset_index()

# Convert 15-min power readings to daily kWh
daily_inverter['total_dc_kwh'] = daily_inverter['total_dc_kwh'] * 0.25
daily_inverter['total_ac_kwh'] = daily_inverter['total_ac_kwh'] * 0.25

# Performance Ratio — compare each inverter to plant daily average
plant_daily_avg = daily_inverter.groupby(['DATE', 'PLANT_LABEL']).agg(
    plant_avg_ac=('total_ac_kwh', 'mean')
).reset_index()

daily_inverter = daily_inverter.merge(plant_daily_avg, on=['DATE', 'PLANT_LABEL'])
daily_inverter['performance_ratio'] = np.where(
    daily_inverter['plant_avg_ac'] > 0,
    daily_inverter['total_ac_kwh'] / daily_inverter['plant_avg_ac'],
    0
)

# Status flags
daily_inverter['status'] = daily_inverter['performance_ratio'].apply(
    lambda x: 'Underperforming' if x < 0.85
    else ('Over-performing' if x > 1.15 else 'Normal')
)

# Lost energy and revenue
PPA_RATE = 40  # $/MWh
daily_inverter['lost_kwh'] = np.where(
    daily_inverter['status'] == 'Underperforming',
    daily_inverter['plant_avg_ac'] - daily_inverter['total_ac_kwh'],
    0
)
daily_inverter['lost_revenue_usd'] = daily_inverter['lost_kwh'] / 1000 * PPA_RATE

print(f"Daily inverter records: {len(daily_inverter):,}")
print(f"\nStatus breakdown:")
print(daily_inverter['status'].value_counts().to_string())
print(f"\nTotal lost energy: {daily_inverter['lost_kwh'].sum():.1f} kWh")
print(f"Total lost revenue: ${daily_inverter['lost_revenue_usd'].sum():.2f}")

In [ ]:
# Daily plant-level summary
daily_plant = daily_inverter.groupby(['DATE', 'PLANT_LABEL', 'PLANT_ID']).agg(
    total_ac_kwh=('total_ac_kwh', 'sum'),
    total_dc_kwh=('total_dc_kwh', 'sum'),
    avg_irradiance=('avg_irradiance', 'mean'),
    avg_ambient_temp=('avg_ambient_temp', 'mean'),
    avg_module_temp=('avg_module_temp', 'mean'),
    avg_performance_ratio=('performance_ratio', 'mean'),
    underperforming_inverters=('status', lambda x: (x == 'Underperforming').sum()),
    total_lost_kwh=('lost_kwh', 'sum'),
    total_lost_revenue=('lost_revenue_usd', 'sum'),
    inverter_count=('SOURCE_KEY', 'nunique')
).reset_index()

daily_plant['DATE'] = pd.to_datetime(daily_plant['DATE'])
daily_plant['rolling_7d_pr'] = daily_plant.groupby('PLANT_LABEL')['avg_performance_ratio'].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)

# Inverter health ranking
inverter_health = daily_inverter.groupby(['PLANT_LABEL', 'SOURCE_KEY']).agg(
    avg_pr=('performance_ratio', 'mean'),
    min_pr=('performance_ratio', 'min'),
    underperforming_days=('status', lambda x: (x == 'Underperforming').sum()),
    total_lost_kwh=('lost_kwh', 'sum'),
    total_lost_revenue=('lost_revenue_usd', 'sum'),
    avg_dc_ac_ratio=('avg_dc_ac_ratio', 'mean')
).reset_index()

inverter_health['health_rank'] = inverter_health.groupby('PLANT_LABEL')['avg_pr'].rank(ascending=True)
inverter_health = inverter_health.sort_values(['PLANT_LABEL', 'health_rank'])

# Alerts
alerts = daily_inverter[daily_inverter['status'] == 'Underperforming'][
    ['DATE', 'PLANT_LABEL', 'PLANT_ID', 'SOURCE_KEY', 'performance_ratio',
     'total_ac_kwh', 'plant_avg_ac', 'lost_kwh', 'lost_revenue_usd', 'status']
].copy()
alerts['DATE'] = pd.to_datetime(alerts['DATE'])

print(f"Daily plant records: {len(daily_plant)}")
print(f"Inverter health records: {len(inverter_health)}")
print(f"Underperformance alerts: {len(alerts)}")

# Show worst inverters
print(f"\n{'='*60}")
print("TOP 5 WORST INVERTERS ACROSS PORTFOLIO")
print(f"{'='*60}")
for _, row in inverter_health.head(5).iterrows():
    print(f"  {row['PLANT_LABEL']} | {row['SOURCE_KEY']} | PR={row['avg_pr']:.3f} | "
          f"Lost={row['total_lost_kwh']:.1f} kWh | Underperforming {row['underperforming_days']:.0f} days")

In [ ]:
# QA CHECKS
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

checks = []

# Check 1: Row counts
c = ("Row count — daily_inverter", f"Expected ~1496, got {len(daily_inverter)}",
     "PASS" if len(daily_inverter) > 1400 else "WARN")
checks.append(c)

# Check 2: No negative power
neg = (daily_inverter['total_ac_kwh'] < 0).sum()
checks.append(("No negative AC power", f"{neg} negative values", "PASS" if neg == 0 else "FAIL"))

# Check 3: PR bounds
oob = ((daily_inverter['performance_ratio'] < 0) | (daily_inverter['performance_ratio'] > 3)).sum()
checks.append(("Performance ratio in bounds (0-3)", f"{oob} out of bounds", "PASS" if oob == 0 else "WARN"))

# Check 4: No future dates
future = (pd.to_datetime(daily_inverter['DATE']) > pd.Timestamp.now()).sum()
checks.append(("No future dates", f"{future} future dates", "PASS" if future == 0 else "FAIL"))

# Check 5: Both plants
plants = daily_inverter['PLANT_LABEL'].nunique()
checks.append(("Both plants present", f"{plants} plants", "PASS" if plants == 2 else "FAIL"))

# Check 6: All inverters
inv = daily_inverter['SOURCE_KEY'].nunique()
checks.append(("All 44 inverters reporting", f"{inv} inverters", "PASS" if inv == 44 else "WARN"))

# Check 7: Temperature bounds
temp_bad = ((daily_inverter['avg_module_temp'] < 10) | (daily_inverter['avg_module_temp'] > 80)).sum()
checks.append(("Temperature in bounds (10-80°C)", f"{temp_bad} out of bounds", "PASS" if temp_bad == 0 else "WARN"))

# Check 8: Lost energy non-negative
neg_lost = (daily_inverter['lost_kwh'] < 0).sum()
checks.append(("Lost energy >= 0", f"{neg_lost} negative values", "PASS" if neg_lost == 0 else "FAIL"))

passed = 0
for check, result, status in checks:
    marker = "✓" if status == "PASS" else "⚠" if status == "WARN" else "✗"
    print(f"{marker} {check:<40} {result:<30} {status}")
    if status == "PASS": passed += 1

print(f"\n{passed}/{len(checks)} checks passed")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
output_dir = '/content/drive/MyDrive/Datasets/Solar/processed'
os.makedirs(output_dir, exist_ok=True)

daily_plant.to_csv(os.path.join(output_dir, 'daily_plant_performance.csv'), index=False)
daily_inverter.to_csv(os.path.join(output_dir, 'daily_inverter_performance.csv'), index=False)
inverter_health.to_csv(os.path.join(output_dir, 'inverter_health_summary.csv'), index=False)
alerts.to_csv(os.path.join(output_dir, 'underperformance_alerts.csv'), index=False)

print("Exported to Google Drive:")
for f in os.listdir(output_dir):
    size = os.path.getsize(os.path.join(output_dir, f))
    print(f"  {f}: {size/1024:.1f} KB")

print(f"\nSummary:")
print(f"  daily_plant_performance: {len(daily_plant)} rows")
print(f"  daily_inverter_performance: {len(daily_inverter)} rows")
print(f"  inverter_health_summary: {len(inverter_health)} rows")
print(f"  underperformance_alerts: {len(alerts)} rows")
print(f"\nTotal: {len(daily_plant) + len(daily_inverter) + len(inverter_health) + len(alerts)} rows")


In [ ]:
print(f"daily_plant: {len(daily_plant)} rows")
print(f"daily_inverter: {len(daily_inverter)} rows")
print(f"inverter_health: {len(inverter_health)} rows")
print(f"alerts: {len(alerts)} rows")

# Dashboard Work:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Solar Asset Performance Dashboard — Plant Comparison', fontsize=16, fontweight='bold', y=1.02)

# Total Energy
plant_summary = daily_plant.groupby('PLANT_LABEL').agg(
    total_kwh=('total_ac_kwh', 'sum'),
    total_lost=('total_lost_kwh', 'sum'),
    avg_underperf=('underperforming_inverters', 'mean')
).reset_index()

colors = ['#2ecc71', '#e74c3c']

axes[0].bar(plant_summary['PLANT_LABEL'], plant_summary['total_kwh'], color=colors)
axes[0].set_title('Total Energy Production (kWh)', fontweight='bold')
axes[0].set_ylabel('kWh')
for i, v in enumerate(plant_summary['total_kwh']):
    axes[0].text(i, v + 50, f'{v:.0f}', ha='center', fontweight='bold')

# Total Lost Energy
axes[1].bar(plant_summary['PLANT_LABEL'], plant_summary['total_lost'], color=colors)
axes[1].set_title('Total Lost Energy (kWh)', fontweight='bold')
axes[1].set_ylabel('kWh')
for i, v in enumerate(plant_summary['total_lost']):
    axes[1].text(i, v + 5, f'{v:.1f}', ha='center', fontweight='bold')

# Avg Underperforming Inverters per Day
axes[2].bar(plant_summary['PLANT_LABEL'], plant_summary['avg_underperf'], color=colors)
axes[2].set_title('Avg Underperforming Inverters/Day', fontweight='bold')
axes[2].set_ylabel('Count')
for i, v in enumerate(plant_summary['avg_underperf']):
    axes[2].text(i, v + 0.2, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Datasets/Solar/processed/panel1_plant_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Panel 1 saved!")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
fig.suptitle('Daily AC Production — Plant 1 vs Plant 2', fontsize=14, fontweight='bold')

for plant, color, marker in [('Plant_1', '#2ecc71', 'o'), ('Plant_2', '#e74c3c', 's')]:
    data = daily_plant[daily_plant['PLANT_LABEL'] == plant]
    ax.plot(data['DATE'], data['total_ac_kwh'], color=color, marker=marker,
            markersize=4, linewidth=1.5, label=plant)

ax.set_xlabel('Date')
ax.set_ylabel('AC Energy (kWh)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Datasets/Solar/processed/panel2_daily_production.png', dpi=150, bbox_inches='tight')
plt.show()
print("Panel 2 saved!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Inverter Health — Performance Ratio by Plant', fontsize=14, fontweight='bold')

for idx, plant in enumerate(['Plant_1', 'Plant_2']):
    data = inverter_health[inverter_health['PLANT_LABEL'] == plant].sort_values('avg_pr')

    colors = ['#e74c3c' if pr < 0.85 else '#f39c12' if pr < 0.95 else '#2ecc71' for pr in data['avg_pr']]

    bars = axes[idx].barh(data['SOURCE_KEY'].str[:8], data['avg_pr'], color=colors)
    axes[idx].set_title(f'{plant} — Inverter Performance', fontweight='bold')
    axes[idx].set_xlabel('Average Performance Ratio')
    axes[idx].axvline(x=0.85, color='red', linestyle='--', alpha=0.7, label='Underperformance Threshold')
    axes[idx].axvline(x=1.0, color='gray', linestyle='--', alpha=0.5, label='Plant Average')
    axes[idx].legend(fontsize=8)
    axes[idx].set_xlim(0.5, 1.3)

    for bar, pr in zip(bars, data['avg_pr']):
        axes[idx].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                      f'{pr:.3f}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Datasets/Solar/processed/panel3_inverter_health.png', dpi=150, bbox_inches='tight')
plt.show()
print("Panel 3 saved!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Lost Energy & Revenue Impact — Plant 2 Top Offenders', fontsize=14, fontweight='bold')

# Top 10 inverters by lost energy
worst = inverter_health[inverter_health['PLANT_LABEL'] == 'Plant_2'].head(10).sort_values('total_lost_kwh', ascending=True)

# Lost kWh
axes[0].barh(worst['SOURCE_KEY'].str[:10], worst['total_lost_kwh'], color='#e74c3c')
axes[0].set_title('Lost Energy (kWh)', fontweight='bold')
axes[0].set_xlabel('kWh Lost')
for i, (_, row) in enumerate(worst.iterrows()):
    axes[0].text(row['total_lost_kwh'] + 0.5, i, f"{row['total_lost_kwh']:.1f}", va='center', fontsize=8)

# Lost Revenue
axes[1].barh(worst['SOURCE_KEY'].str[:10], worst['total_lost_revenue'], color='#e67e22')
axes[1].set_title('Lost Revenue (USD)', fontweight='bold')
axes[1].set_xlabel('USD Lost')
for i, (_, row) in enumerate(worst.iterrows()):
    axes[1].text(row['total_lost_revenue'] + 0.02, i, f"${row['total_lost_revenue']:.2f}", va='center', fontsize=8)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Datasets/Solar/processed/panel4_lost_energy_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print("Panel 4 saved!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Weather Impact — Irradiance vs AC Production', fontsize=14, fontweight='bold')

for idx, plant in enumerate(['Plant_1', 'Plant_2']):
    data = daily_plant[daily_plant['PLANT_LABEL'] == plant]

    scatter = axes[idx].scatter(data['avg_irradiance'], data['total_ac_kwh'],
                                c=data['avg_module_temp'], cmap='RdYlGn_r',
                                s=60, alpha=0.8, edgecolors='black', linewidth=0.5)

    # Trend line
    z = np.polyfit(data['avg_irradiance'], data['total_ac_kwh'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(data['avg_irradiance'].min(), data['avg_irradiance'].max(), 100)
    axes[idx].plot(x_line, p(x_line), '--', color='black', alpha=0.5, label=f'Trend (slope={z[0]:.0f})')

    axes[idx].set_title(f'{plant}', fontweight='bold')
    axes[idx].set_xlabel('Avg Irradiance (kWh/m²)')
    axes[idx].set_ylabel('Total AC Energy (kWh)')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(True, alpha=0.3)

plt.colorbar(scatter, ax=axes, label='Module Temp (°C)', shrink=0.8)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Datasets/Solar/processed/panel5_weather_vs_production.png', dpi=150, bbox_inches='tight')
plt.show()
print("Panel 5 saved!")